# Step 06: Merge Institution information

Input:

    - Publication, Journal, Gender, Affiliation
        - data/interim/publication/5_first_author_publication_info.csv
        - data/interim/publication/5_last_author_publication_info.csv

    - Institution
        - data/raw/institution/1_ARWU_institution_list.csv
        - data/raw/institution/2_QS_institution_list.xlsx
        - data/raw/institution/3_THE_institution_list.xlsx
        - data/raw/institution/4_OpenAlex.csv
        
Output:
        - data/interim/publication/6_first_author_institution_info.csv
        - data/interim/publication/6_last_author_institution_info.csv


Purpose:
    - Merge insitution's ranking information

In [ ]:
import pandas as pd
import numpy as np
from rapidfuzz import process
from joblib import Parallel, delayed
import joblib
from tqdm import tqdm
import sys
import os

In [ ]:
# -----------------------------
# Paths
# -----------------------------


PROJECT_ROOT = Path(__file__).resolve().parents[2]
FIRST_IN = PROJECT_ROOT / "data" / "interim" / "publication"/ "5_first_author_publication_info.csv"
LAST_IN = PROJECT_ROOT / "data" / "interim" / "publication"/ "5_last_author_publication_info.csv"

ARWU_IN = PROJECT_ROOT / "data" / "raw" / "institution" / "1_ARWU_institution_list.csv"
QS_IN = PROJECT_ROOT / "data" / "raw" / "institution" / "2_QS_institution_list.xlsx"
THE_IN = PROJECT_ROOT / "data" / "raw" / "institution" / "3_THE_institution_list.xlsx"
OPENALEX_IN = PROJECT_ROOT / "data" / "raw" / "institution" / "4_OpenAlex.csv"


FIRST_OUT = PROJECT_ROOT / "data" / "interim" / "publication"/ "6_first_author_institution_info.csv"
LAST_OUT = PROJECT_ROOT / "data" / "interim" / "publication"/ "6_last_author_institution_info.csv"

In [ ]:
# First authorship
use_col = ['index_num','affiliation','Publication Year']

first = pd.read_csv(FIRST_IN, usecols=use_col)
last = pd.read_csv(LAST_IN, usecols=use_col)



# 1 ARWU ranking

In [ ]:
# read the CSV file into a Dataframe
arwu = pd.read_csv(ARWU_IN)

arwu.rename(columns={
    'Institution': 'affiliation',
    'World Rank': 'ins_arwu',
    'year': 'Publication Year'
}, inplace=True)

arwu

In [ ]:
# Fuzzy match with first authorship

def preprocess_arwu(rank):
    arwu_dict = {}
    for year, group in rank.groupby('Publication Year'):
        arwu_dict[year] = group[['affiliation','ins_arwu']].dropna().values
    return arwu_dict

# 定义模糊匹配的函数
def fuzzy_match(university, year, arwu_dict, cache):
    if year not in arwu_dict:
        return None

    if university in cache:
        return cache[university]

    match = process.extractOne(university, arwu_dict[year][:, 0], score_cutoff=90)
    if match:
        result = arwu_dict[year][match[2], 1]
        cache[university] = result
        return result
    return None

# 主处理函数
def process_batch(first, arwu_dict):
    cache = {}
    universities = first['affiliation'].values
    years = first['Publication Year'].values
    return [fuzzy_match(univ, year, arwu_dict, cache) for univ, year in zip(universities, years)]

# 主程序
arwu_dict = preprocess_arwu(arwu)

batch_size = 1000000
num_cores = joblib.cpu_count()

results = []
for start in tqdm(range(0, len(first), batch_size), file=sys.stdout):
    end = min(start + batch_size, len(first))
    batch_data = first.iloc[start:end]

    batch_results = Parallel(n_jobs=num_cores)(
        delayed(process_batch)(batch_data[i:i+100000], arwu_dict)
        for i in range(0, len(batch_data), 100000)
    )
    results.extend([item for sublist in batch_results for item in sublist])

first['ins_arwu'] = results

In [ ]:
# Fuzzy match with last authorship

def preprocess_arwu(rank):
    arwu_dict = {}
    for year, group in rank.groupby('Publication Year'):
        arwu_dict[year] = group[['affiliation','ins_arwu']].dropna().values
    return arwu_dict

# 定义模糊匹配的函数
def fuzzy_match(university, year, arwu_dict, cache):
    if year not in arwu_dict:
        return None

    if university in cache:
        return cache[university]

    match = process.extractOne(university, arwu_dict[year][:, 0], score_cutoff=90)
    if match:
        result = arwu_dict[year][match[2], 1]
        cache[university] = result
        return result
    return None

# 主处理函数
def process_batch(last, arwu_dict):
    cache = {}
    universities = last['affiliation'].values
    years = last['Publication Year'].values
    return [fuzzy_match(univ, year, arwu_dict, cache) for univ, year in zip(universities, years)]

# 主程序
arwu_dict = preprocess_arwu(arwu)

batch_size = 1000000
num_cores = joblib.cpu_count()

results = []
for start in tqdm(range(0, len(last), batch_size), file=sys.stdout):
    end = min(start + batch_size, len(last))
    batch_data = last.iloc[start:end]

    batch_results = Parallel(n_jobs=num_cores)(
        delayed(process_batch)(batch_data[i:i+100000], arwu_dict)
        for i in range(0, len(batch_data), 100000)
    )
    results.extend([item for sublist in batch_results for item in sublist])

last['ins_arwu'] = results

# 2 QS ranking

In [ ]:
qs = pd.read_excel(QS_IN)

print(qs.columns)
print(qs.head())

qs = qs[['年份', '学校名称', '排名','Academic Reputation','Citations per Faculty','Employer Reputation']]
qs.loc[:, '排名'] = qs['排名'].replace(9999, np.nan)

qs.rename(columns={
    '学校名称': 'affiliation',
    '排名': 'ins_qs',
    '年份': 'Publication Year',
    'Academic Reputation':'ins_AcademicRep',
    'Citations per Faculty':'ins_CiteFaculty',
    'Employer Reputation':'ins_EmployerRep'
}, inplace=True)

qs = qs.dropna(subset=['affiliation'], how='any')

qs['qs_id'] = qs.groupby(['affiliation', 'Publication Year']).ngroup()
qs = qs.dropna(subset=['qs_id', 'ins_qs'], how='any')
print(qs['qs_id'].isna().sum())   

print(qs.head())

qs_id = qs[['affiliation', 'Publication Year','qs_id']]
qs_id

In [ ]:
# Fuzzy match with first authorship
def preprocess_rank(rank):
    arwu_dict = {}
    for year, group in rank.groupby('Publication Year'):
        arwu_dict[year] = group[['affiliation','qs_id']].dropna().values
    return arwu_dict

# 定义模糊匹配的函数
def fuzzy_match(university, year, arwu_dict, cache):
    if year not in arwu_dict:
        return None

    if university in cache:
        return cache[university]

    match = process.extractOne(university, arwu_dict[year][:, 0], score_cutoff=95)
    if match:
        result = arwu_dict[year][match[2], 1]
        cache[university] = result
        return result
    return None

# 主处理函数
def process_batch(first, arwu_dict):
    cache = {}
    universities = first['affiliation'].values
    years = first['Publication Year'].values
    return [fuzzy_match(univ, year, arwu_dict, cache) for univ, year in zip(universities, years)]

# 主程序
arwu_dict = preprocess_rank(qs_id)

batch_size = 1000000
num_cores = joblib.cpu_count()

results = []
for start in tqdm(range(0, len(first), batch_size), file=sys.stdout):
    end = min(start + batch_size, len(first))
    batch_data = first.iloc[start:end]

    batch_results = Parallel(n_jobs=num_cores)(
        delayed(process_batch)(batch_data[i:i+100000], arwu_dict)
        for i in range(0, len(batch_data), 100000)
    )
    results.extend([item for sublist in batch_results for item in sublist])

first['qs_id'] = results

In [ ]:
first = first.merge(qs,on='qs_id',how='left')
first

first.rename(columns={
    'affiliation_x': 'affiliation',
    'Publication Year_x': 'Publication Year',
    'affiliation_y': 'affiliation_qs'
}, inplace=True)

first = first.drop(['Publication Year_y','qs_id'], axis=1)

first

In [ ]:
# Fuzzy match with last authorship

def preprocess_rank(rank):
    arwu_dict = {}
    for year, group in rank.groupby('Publication Year'):
        arwu_dict[year] = group[['affiliation','qs_id']].dropna().values
    return arwu_dict

# 定义模糊匹配的函数
def fuzzy_match(university, year, arwu_dict, cache):
    if year not in arwu_dict:
        return None

    if university in cache:
        return cache[university]

    match = process.extractOne(university, arwu_dict[year][:, 0], score_cutoff=95)
    if match:
        result = arwu_dict[year][match[2], 1]
        cache[university] = result
        return result
    return None

# 主处理函数
def process_batch(last, arwu_dict):
    cache = {}
    universities = last['affiliation'].values
    years = last['Publication Year'].values
    return [fuzzy_match(univ, year, arwu_dict, cache) for univ, year in zip(universities, years)]

# 主程序
arwu_dict = preprocess_rank(qs_id)

batch_size = 1000000
num_cores = joblib.cpu_count()

results = []
for start in tqdm(range(0, len(last), batch_size), file=sys.stdout):
    end = min(start + batch_size, len(last))
    batch_data = last.iloc[start:end]

    batch_results = Parallel(n_jobs=num_cores)(
        delayed(process_batch)(batch_data[i:i+100000], arwu_dict)
        for i in range(0, len(batch_data), 100000)
    )
    results.extend([item for sublist in batch_results for item in sublist])

last['qs_id'] = results

In [ ]:
last = last.merge(qs,on='qs_id',how='left')

last = last.drop(['Publication Year_y','qs_id'], axis=1)
last.rename(columns={
    'affiliation_x': 'affiliation',
    'Publication Year_x': 'Publication Year',
    'affiliation_y': 'affiliation_qs'
}, inplace=True)

last


# 3 THE rank

In [ ]:
# read the CSV file into a Dataframe
the = pd.read_excel(THE_IN)

print(the.columns)
print(the.head())

the = the.dropna(subset=['排名','学校名称'], how='any')
the

In [ ]:
# 1. Drop rows where 排名 == "Reporter"
the = the[the['排名'] != "Reporter"].copy()

# 2. Remove leading "=" sign if exists
the['排名'] = the['排名'].str.lstrip("=")

# 3. Handle ranges like "226-250" → take the first number
the['排名'] = the['排名'].str.replace(r"(\d+)-\d+", r"\1", regex=True)
the['排名'] = the['排名'].str.replace(r"(\d+)–\d+", r"\1", regex=True)

# 4. Handle "801+" → take the number before "+"
the['排名'] = the['排名'].str.replace(r"(\d+)\+", r"\1", regex=True)

# 5. Convert to numeric
the['排名'] = pd.to_numeric(the['排名'], errors='coerce')

In [ ]:
# List of affected columns
cols = ["scores_industry_income", "scores_international_outlook","scores_overall"]

the['scores_industry_income'] = the['scores_industry_income'].str.replace(r"(\d+)-\d+", r"\1", regex=True)
the['scores_international_outlook'] = the['scores_international_outlook'].str.replace(r"(\d+)-\d+", r"\1", regex=True)

# List of affected columns
cols = ["scores_industry_income", "scores_international_outlook"]

# Replace ',' with '.' and convert to float
for col in cols:
    the[col] = (
        the[col]
        .astype(str)            # make sure they're strings
        .str.replace("-", "NaN", regex=False)
        .astype(float)          # convert to float
    )


the = the.drop(['scores_overall','地区'], axis=1)

the.rename(columns={
    '学校名称': 'affiliation',
    '年份': 'Publication Year',
    '排名': 'ins_therank',
    'scores_citations':'ins_the_citations',
    'scores_industry_income':'ins_the_indinc',
    'scores_international_outlook':'ins_the_intlotl',
    'scores_research':'ins_the_research',
    'scores_teaching':'ins_the_teaching'
}, inplace=True)

the['the_id'] = the.groupby(['affiliation', 'Publication Year']).ngroup()

In [ ]:
the_id = the[['affiliation', 'Publication Year','the_id']]
the_id

In [ ]:
# Fuzzy match with first authorship

def preprocess_rank(rank):
    arwu_dict = {}
    for year, group in rank.groupby('Publication Year'):
        arwu_dict[year] = group[['affiliation','the_id']].dropna().values
    return arwu_dict

# 定义模糊匹配的函数
def fuzzy_match(university, year, arwu_dict, cache):
    if year not in arwu_dict:
        return None

    if university in cache:
        return cache[university]

    match = process.extractOne(university, arwu_dict[year][:, 0], score_cutoff=95)
    if match:
        result = arwu_dict[year][match[2], 1]
        cache[university] = result
        return result
    return None

# 主处理函数
def process_batch(first, arwu_dict):
    cache = {}
    universities = first['affiliation'].values
    years = first['Publication Year'].values
    return [fuzzy_match(univ, year, arwu_dict, cache) for univ, year in zip(universities, years)]

# 主程序
arwu_dict = preprocess_rank(the_id)

batch_size = 1000000
num_cores = joblib.cpu_count()

results = []
for start in tqdm(range(0, len(first), batch_size), file=sys.stdout):
    end = min(start + batch_size, len(first))
    batch_data = first.iloc[start:end]

    batch_results = Parallel(n_jobs=num_cores)(
        delayed(process_batch)(batch_data[i:i+100000], arwu_dict)
        for i in range(0, len(batch_data), 100000)
    )
    results.extend([item for sublist in batch_results for item in sublist])

first['the_id'] = results

In [ ]:
first = first.merge(the,on='the_id',how='left')

first = first.drop(['Publication Year_y','the_id'], axis=1)
first.rename(columns={
    'affiliation_x': 'affiliation',
    'Publication Year_x': 'Publication Year',
    'affiliation_y': 'affiliation_the'
}, inplace=True)
first

In [ ]:
# Fuzzy match with last authorship

def preprocess_rank(rank):
    arwu_dict = {}
    for year, group in rank.groupby('Publication Year'):
        arwu_dict[year] = group[['affiliation','the_id']].dropna().values
    return arwu_dict

# 定义模糊匹配的函数
def fuzzy_match(university, year, arwu_dict, cache):
    if year not in arwu_dict:
        return None

    if university in cache:
        return cache[university]

    match = process.extractOne(university, arwu_dict[year][:, 0], score_cutoff=95)
    if match:
        result = arwu_dict[year][match[2], 1]
        cache[university] = result
        return result
    return None

# 主处理函数
def process_batch(last, arwu_dict):
    cache = {}
    universities = last['affiliation'].values
    years = last['Publication Year'].values
    return [fuzzy_match(univ, year, arwu_dict, cache) for univ, year in zip(universities, years)]

# 主程序
arwu_dict = preprocess_rank(the_id)

batch_size = 1000000
num_cores = joblib.cpu_count()

results = []
for start in tqdm(range(0, len(last), batch_size), file=sys.stdout):
    end = min(start + batch_size, len(last))
    batch_data = last.iloc[start:end]

    batch_results = Parallel(n_jobs=num_cores)(
        delayed(process_batch)(batch_data[i:i+100000], arwu_dict)
        for i in range(0, len(batch_data), 100000)
    )
    results.extend([item for sublist in batch_results for item in sublist])

last['the_id'] = results

In [ ]:
last = last.merge(the,on='the_id',how='left')
last = last.drop(['Publication Year_y','the_id'], axis=1)
last.rename(columns={
    'affiliation_x': 'affiliation',
    'Publication Year_x': 'Publication Year',
    'affiliation_y': 'affiliation_the'
}, inplace=True)
last

# 4 OpenAlex

In [ ]:
# Load institution data
institution_df = pd.read_csv(OPENALEX_IN)

print(f"Institution data shape: {institution_df.shape}")

# Keep only required variables from institution data and rename with "ins_" prefix
institution_cols = ["year", "works_count", "cited_by_count","display_name","clean_institution_id"]
oa = institution_df[institution_cols].copy()

# Rename columns with "ins_" prefix (except for matching columns)
rename_dict = {
    "year":"Publication Year",
    "display_name":"affiliation",
    "works_count": "ins_oa_wrkcnt",
    "cited_by_count": "ins_oa_ctdbycnt",
    "clean_institution_id":"ins_oa_id"
}
oa = oa.rename(columns=rename_dict)

print("Institution data columns after renaming:", oa.columns.tolist())
del institution_df

oa = oa.dropna(subset=['ins_oa_wrkcnt','ins_oa_ctdbycnt'], how='any')
oa['oa_id'] = oa.groupby(['ins_oa_id', 'Publication Year']).ngroup()

oa_id = oa[['affiliation', 'Publication Year','oa_id']]

In [ ]:
# Fuzzy match with first authorship

def preprocess_rank(rank):
    arwu_dict = {}
    for year, group in rank.groupby('Publication Year'):
        arwu_dict[year] = group[['affiliation','oa_id']].dropna().values
    return arwu_dict

# 定义模糊匹配的函数
def fuzzy_match(university, year, arwu_dict, cache):
    if year not in arwu_dict:
        return None

    if university in cache:
        return cache[university]

    match = process.extractOne(university, arwu_dict[year][:, 0], score_cutoff=95)
    if match:
        result = arwu_dict[year][match[2], 1]
        cache[university] = result
        return result
    return None

# 主处理函数
def process_batch(first, arwu_dict):
    cache = {}
    universities = first['affiliation'].values
    years = first['Publication Year'].values
    return [fuzzy_match(univ, year, arwu_dict, cache) for univ, year in zip(universities, years)]

# 主程序
arwu_dict = preprocess_rank(oa_id)

batch_size = 1000000
num_cores = joblib.cpu_count()

results = []
for start in tqdm(range(0, len(first), batch_size), file=sys.stdout):
    end = min(start + batch_size, len(first))
    batch_data = first.iloc[start:end]

    batch_results = Parallel(n_jobs=num_cores)(
        delayed(process_batch)(batch_data[i:i+100000], arwu_dict)
        for i in range(0, len(batch_data), 100000)
    )
    results.extend([item for sublist in batch_results for item in sublist])

first['oa_id'] = results

In [ ]:
first = first.merge(oa,on='oa_id',how='left')

first = first.drop(['Publication Year_y','oa_id','ins_oa_id'], axis=1)

first.rename(columns={
    'affiliation_x': 'affiliation',
    'Publication Year_x': 'Publication Year',
    'affiliation_y': 'affiliation_oa'
}, inplace=True)

first


In [ ]:
# fuzzy match with last authorship

def preprocess_rank(rank):
    arwu_dict = {}
    for year, group in rank.groupby('Publication Year'):
        arwu_dict[year] = group[['affiliation','oa_id']].dropna().values
    return arwu_dict

# 定义模糊匹配的函数
def fuzzy_match(university, year, arwu_dict, cache):
    if year not in arwu_dict:
        return None

    if university in cache:
        return cache[university]

    match = process.extractOne(university, arwu_dict[year][:, 0], score_cutoff=95)
    if match:
        result = arwu_dict[year][match[2], 1]
        cache[university] = result
        return result
    return None

# 主处理函数
def process_batch(last, arwu_dict):
    cache = {}
    universities = last['affiliation'].values
    years = last['Publication Year'].values
    return [fuzzy_match(univ, year, arwu_dict, cache) for univ, year in zip(universities, years)]

# 主程序
arwu_dict = preprocess_rank(oa_id)

batch_size = 1000000
num_cores = joblib.cpu_count()

results = []
for start in tqdm(range(0, len(last), batch_size), file=sys.stdout):
    end = min(start + batch_size, len(last))
    batch_data = last.iloc[start:end]

    batch_results = Parallel(n_jobs=num_cores)(
        delayed(process_batch)(batch_data[i:i+100000], arwu_dict)
        for i in range(0, len(batch_data), 100000)
    )
    results.extend([item for sublist in batch_results for item in sublist])

last['oa_id'] = results

In [ ]:
last = last.merge(oa,on='oa_id',how='left')
last = last.drop(['Publication Year_y','oa_id','ins_oa_id'], axis=1)
last.rename(columns={
    'affiliation_x': 'affiliation',
    'Publication Year_x': 'Publication Year',
    'affiliation_y': 'affiliation_oa'
}, inplace=True)
last

In [ ]:

first.to_csv(FIRST_OUT, index=False)